# 62 — Validate the Dataset Specialization A-Box against the published shapes

The A-Box **does not conform**, and that is the deliverable of this notebook
rather than a problem with it (decision D11) — the same stance `60_` takes for
the concept layer.

Every violation traces to one of five causes. What this notebook adds over
`60_` is that each cause has an *expected count* computed from the data graph
itself — the number of `rdf:_n` edges, the number of `role` values, the number
of IRI-valued `dataElementConceptId` edges — and the notebook asserts that the
validator's count equals it, per domain. So a violation is not merely explained;
it is accounted for exactly, and a new `NEW_` reference or an enum value the
T-Box does not know shows up as a mismatch rather than as a count that drifted.

Requires `pyshacl`. The 32 files validate in under a minute.

## Configuration

In [ ]:
ROOT    = ".."
REPORTS = "../reports"
DSS_DIR = "../dss"

SHAPES  = f"{ROOT}/cosmos_sdtm_v1.shapes.ttl"
SDTM_NS = "https://www.cdisc.org/cosmos/sdtm_v1.0/"

ORDER      = "authored: rdf:_n carries the D5 variable order on the closed SDTMGroup shape (D27)"
IDENTITY   = "authored: dcterms:identifier carries the mnemonic or the codelist C-code (D3, D29)"
ANCHOR     = "authored: skos:exactMatch to the EVS form on the codelist node (D29, D2)"
ENUM       = "generator disagreement: the enum has no meaning:, so gen-owl declares class IRIs and gen-shacl expects strings (known-gaps 4b)"
REFERENCE  = "published range is string; the reference is rendered as an edge to the NCIt node (D21, D28, D31)"

# Every expected (constraint, path) pair, and why it happens. rdf:_1, rdf:_2, ...
# are normalised to "_n" before lookup. A result outside this map fails the notebook.
EXPECTED = {
    ("ClosedConstraintComponent", "_n"): ORDER,
    ("ClosedConstraintComponent", "identifier"): IDENTITY,
    ("ClosedConstraintComponent", "exactMatch"): ANCHOR,
}
for slot in ("role", "predicateTerm", "linkingPhrase", "dataType", "comparator", "packageType"):
    EXPECTED[("InConstraintComponent", slot)] = ENUM
for slot in ("dataElementConceptId", "conceptId", "biomedicalConceptId"):
    for component in ("DatatypeConstraintComponent", "NodeKindConstraintComponent", "PatternConstraintComponent"):
        EXPECTED[(component, slot)] = REFERENCE

## What each cause should count to

The expected count of a cause is a query over the data graph, not a number
typed in.

- `rdf:_n` closed-shape results: one per membership edge, which is one per
  variable.
- `identifier` closed-shape results: one per specialization plus one per
  codelist node in the file; `exactMatch`: one per codelist node. Codelist nodes
  are repeated across domain files (D32), so these are counted per file.
- enum results: one per value of that slot. `originType` and `originSource`
  are absent from the map on purpose — the two enums that carry `meaning:` in
  the published model conform, because both generators then use the NCIt IRI.
  Six do not; that is the measurement behind the ask in `known-gaps.md` §4b.
- reference results: three per IRI-valued edge (datatype, node kind, pattern).
  The five references carried as literals under D31 conform to the published
  pattern and produce nothing — asserted by the same count.

In [ ]:
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import DCTERMS, RDF, SKOS

SDTM = Namespace(SDTM_NS)
RDF_MEMBER = str(RDF) + "_"


def expected_counts(data):
    """(constraint, path) -> count the validator should report on this graph."""
    groups = set(data.subjects(RDF.type, SDTM.SDTMGroup))
    codelists = set(data.subjects(RDF.type, SDTM.CodeList))
    members = sum(1 for _, p, _ in data if str(p).startswith(RDF_MEMBER))

    counts = {
        ("ClosedConstraintComponent", "_n"): members,
        ("ClosedConstraintComponent", "identifier"): len(groups) + len(codelists),
        ("ClosedConstraintComponent", "exactMatch"): len(codelists),
    }
    for slot in ("role", "predicateTerm", "linkingPhrase", "dataType", "comparator", "packageType"):
        counts[("InConstraintComponent", slot)] = sum(1 for _ in data.objects(None, SDTM[slot]))
    for slot in ("dataElementConceptId", "conceptId", "biomedicalConceptId"):
        edges = sum(1 for o in data.objects(None, SDTM[slot]) if isinstance(o, URIRef))
        for component in ("DatatypeConstraintComponent", "NodeKindConstraintComponent", "PatternConstraintComponent"):
            counts[(component, slot)] = edges
    return counts


if set(expected_counts(Graph())) != set(EXPECTED):
    raise RuntimeError("the expected-count function and the cause map do not cover the same pairs")
print(f"{len(EXPECTED)} (constraint, path) pairs, {len(set(EXPECTED.values()))} causes")

## Validate every domain file

`local(path)` reduces a predicate IRI to its last segment, which is enough to
group by — the shapes and the data share one vocabulary — and folds every
`rdf:_1`, `rdf:_2`, … onto `_n`.

In [ ]:
from collections import Counter
from pathlib import Path

from pyshacl import validate
from rdflib.namespace import SH

shapes = Graph().parse(SHAPES, format="turtle")
files = sorted(Path(DSS_DIR).glob("cosmos_sdtm_v1.*.instances.ttl"))
if len(files) != 32:
    raise RuntimeError(f"expected 32 domain files under {DSS_DIR}, found {len(files)}")


def local(term):
    if term is None:
        return "-"
    text = str(term)
    segment = text.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
    if segment.startswith("_") and segment[1:].isdigit():
        return "_n"
    return segment


rows = []
unexplained = []
mismatches = []
totals = Counter()

for file in files:
    domain = file.name.split(".")[1]
    data = Graph().parse(file, format="turtle")
    conforms, results, _ = validate(data, shacl_graph=shapes, inference="none", advanced=True)

    actual = Counter()
    for result in results.subjects(RDF.type, SH.ValidationResult):
        key = (local(results.value(result, SH.sourceConstraintComponent)),
               local(results.value(result, SH.resultPath)))
        if key not in EXPECTED:
            unexplained.append((domain, key, str(results.value(result, SH.focusNode))))
        actual[key] += 1

    expected = expected_counts(data)
    for key in EXPECTED:
        if actual[key] != expected[key]:
            mismatches.append((domain, key, actual[key], expected[key]))
        if actual[key]:
            rows.append({"domain": domain, "constraint": key[0], "path": key[1],
                         "cause": EXPECTED[key], "count": actual[key]})
    totals.update(actual)

    print(f"{domain}  {len(data):>7,} triples  conforms={conforms!s:5}  {sum(actual.values()):>7,} results")

print()
print(f"{sum(totals.values()):,} results over {len(files)} files; {len(unexplained)} unexplained; {len(mismatches)} count mismatches")

## Totals by cause

In [ ]:
by_cause = Counter()
for key, n in totals.items():
    by_cause[EXPECTED.get(key, "UNEXPLAINED")] += n

for (component, path), n in sorted(totals.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"{n:>8,}  {component:30s} {path:22s} {EXPECTED.get((component, path), 'UNEXPLAINED')[:60]}")
print()
for cause, n in by_cause.most_common():
    print(f"{n:>8,}  {cause}")

## Write the summary report

One row per (domain, constraint, path): the count and its cause. No per-focus
file — every result is accounted for by the arithmetic above, and a 117,000-row
listing would add bytes without information.

In [ ]:
import csv

summary = Path(REPORTS, "dss_shacl_conformance_summary.csv")
with open(summary, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["domain", "constraint", "path", "cause", "count"])
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda r: (r["domain"], r["constraint"], r["path"])))

print(f"wrote {summary}  ({len(rows)} rows)")

## Result

Non-conformance is expected and recorded. What is **not** acceptable is a
violation nobody has accounted for, or a count that does not match the graph.

In [ ]:
if unexplained:
    for u in unexplained[:10]:
        print("   ", u)
    raise RuntimeError(
        f"{len(unexplained):,} validation result(s) fall outside the five causes. "
        "An unexplained violation is either a rendering bug or a new finding, and both matter.")
if mismatches:
    for m in mismatches[:10]:
        print("   ", m)
    raise RuntimeError(
        f"{len(mismatches)} (domain, constraint, path) count(s) differ from what the data graph predicts. "
        "Either the rendering changed shape or a cause has a case this notebook does not know.")

print(f"{sum(totals.values()):,} violations over 32 files, all five causes accounted for exactly.")
print()
print("The A-Box does not conform to the published shapes, by design:")
print("  - one cause is the D5 order carried as rdf:_n on a closed shape (D27)")
print("  - two are this repo adding identity the schema has no slot for (D3, D29)")
print("  - one is gen-owl and gen-shacl disagreeing about what an enum value is - for the six enums without meaning:")
print("  - one is a reference the published model types as a string, rendered as an edge (D21, D28, D31)")